In [ ]:
import pandas as pd
df=pd.read_csv('seoul_air_quality_2022_english.csv')
df_raw=df.copy()
pd.set_option('display.max_columns',500)
pd.set_option('display.max_rows',500)
pd.set_option('display.max_colwidth',500)
df.head(500)
print(df.shape)

In [ ]:
df.info()
df.describe()
df.isnull().sum()
null_percentage=(df.isnull().sum())*100/len(df)
print(null_percentage)

In [ ]:
df['datetime']=pd.to_datetime(df['datetime'])
df['hour']=df['datetime'].dt.hour
df['month']=df['datetime'].dt.month
df['date']=df['datetime'].dt.date
pollutants=['CO','NO2','O3','PM10','PM2.5','SO2']
def get_season(month):
    if month in [12,1,2]:
        return "Winter"
    elif month in [3,4,5]:
        return "Spring"
    elif month in [6,7,8]:
        return "Summer"
    else:
        return "Autumn"
df['season']=df['month'].apply(get_season)
for pollutant in pollutants:
    df[pollutant]=df[pollutant].fillna(df.groupby(['station_code','hour','season'])[pollutant].transform('median'))


In [ ]:
#unit normalization
df['NO2_ugm3']=df['NO2']*1880
df['SO2_ugm3']=df['SO2']*2620
df['O3_ugm3']=df['O3']*1960
df['CO_ugm3']=df['CO']*1145

In [ ]:
pollutant_category=df[(df['PM10']>45)|(df['PM2.5']>15)|(df['NO2_ugm3']>25)]
daily_df=df.groupby(['district','date']).agg(PM25=('PM2.5','mean'),PM10=('PM10','mean'),NO2=('NO2_ugm3','mean'),population=('population','first'),po_density=('population_density','first'),area=('area_km2','first')).reset_index()
pollution=daily_df[(daily_df['PM25']>15)|(daily_df['PM10']>45)|(daily_df['NO2']>25)]
district_summary=pollution.groupby(['district']).agg(total_days=('date','count'),district_population=('population','first'),district_popdens=('po_density','first'),dis_area=('area','first')).reset_index()
median_seoul=district_summary['total_days'].median()
serious_pollution=district_summary[district_summary['total_days']>median_seoul]
serious_pollution=serious_pollution.sort_values(by='total_days',ascending=False)
print(f'Median Seoul: {median_seoul} days')
print(serious_pollution)
#Demographic Exposure
dens_seoul=district_summary['district_popdens'].median()
pop_seoul=district_summary['district_population'].median()
filtered_district=district_summary.query('district_population>@pop_seoul|district_popdens>@dens_seoul')
print(filtered_district)
#Area Filter
med_seoularea=district_summary['dis_area'].median()
filtered_area=district_summary.query('dis_area>@med_seoularea')